In [1]:
import pandas as pd
import numpy as np
import joblib
import pickle
import os
import sys
import pytz
import datetime
import torch
from pathlib import Path
import utils.gcn_network.network as net
from utils.model_config.input_config import SYNDICATE_CONFIG, DBENVS
from utils.data_extraction.data_extraction import data_extraction
from utils.data_transformation.data_transformation import main as data_transformation, connection_process
from utils.community_detection.graph_construction import main as graph_construction
from utils.community_detection.model_scoring import main as model_scoring
from utils.data_output.edh_writing import main as edh_upload_data
# from utils.data_output.edh_writing import extract_scored_data
from utils.model_logging.model_logging import log_utility

An error occurred: module 'importlib.metadata' has no attribute 'packages_distributions'


c:\GitHub\CTP_syndicate_model\.venv\lib\site-packages\google\api_core\_python_version_support.py:252: FutureWarning: You are using a Python version (3.9.13) past its end of life. Google will update google.api_core with critical bug fixes on a best-effort basis, but not with any other fixes or features. Please upgrade to the latest Python version, or at least Python 3.10, and then update google.api_core.
  warnings.warn(message, FutureWarning)


In [2]:
scoring_config = SYNDICATE_CONFIG        
node_data, doc_lawyer_df, doc_psych_df, doc_repair_df, vehicle_df = data_extraction(scoring_config)
df_node_feature, edges_grouped = data_transformation(scoring_config, node_data, doc_lawyer_df, doc_psych_df, doc_repair_df, vehicle_df)

{"model_id": "AMM-395", "component": "data_extraction", "pipeline_job": null, "event": "Starting data extraction process", "logger": "utils.model_logging.model_logging", "level": "info", "timestamp": "2025-12-03T04:04:24.393847Z"}
{"model_id": "AMM-395", "component": "data_extraction", "pipeline_job": null, "event": "Data extraction date range: 2018-01-01 to 2025-12-03", "logger": "utils.model_logging.model_logging", "level": "info", "timestamp": "2025-12-03T04:04:24.397026Z"}
{"model_id": "AMM-395", "component": "data_extraction", "pipeline_job": null, "event": "SQL execution completed", "logger": "utils.model_logging.model_logging", "level": "info", "timestamp": "2025-12-03T04:06:34.327033Z"}
{"model_id": "AMM-395", "component": "data_extraction", "pipeline_job": null, "event": "Data extraction completed", "logger": "utils.model_logging.model_logging", "level": "info", "timestamp": "2025-12-03T04:06:34.327033Z"}
{"model_id": "AMM-395", "component": "data_transformation", "pipeline_jo

In [3]:
graph_data, G, all_nodes_df, final_communities = graph_construction(scoring_config, df_node_feature, edges_grouped)

{"model_id": "AMM-395", "component": "community_detection", "pipeline_job": null, "event": "Starting graph construction and community detection", "logger": "utils.model_logging.model_logging", "level": "info", "timestamp": "2025-12-03T04:06:35.662142Z"}


c:\GitHub\CTP_syndicate_model\model_package\model_scoring\utils\community_detection\graph_construction.py:74: UserWarning: Creating a tensor from a list of numpy.ndarrays is extremely slow. Please consider converting the list to a single numpy.ndarray with numpy.array() before converting to a tensor. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\builder\windows\pytorch\torch\csrc\utils\tensor_new.cpp:281.)
  edge_index = torch.tensor([edges_grouped['source_idx'].values, edges_grouped['target_idx'].values], dtype=torch.long)


{"model_id": "AMM-395", "component": "community_detection", "pipeline_job": null, "event": "Graph construction completed", "logger": "utils.model_logging.model_logging", "level": "info", "timestamp": "2025-12-03T04:06:37.441924Z"}
Loading data locally: C:\GitHub\CTP_syndicate_model\model_package\model_scoring\trained_models\communities_init.pkl
{"model_id": "AMM-395", "component": "community_detection", "pipeline_job": null, "event": "Using warm start with 786 initial communities", "logger": "utils.model_logging.model_logging", "level": "info", "timestamp": "2025-12-03T04:06:37.475123Z"}
{"model_id": "AMM-395", "component": "community_detection", "pipeline_job": null, "event": "Community detection completed: 766 communities detected", "logger": "utils.model_logging.model_logging", "level": "info", "timestamp": "2025-12-03T04:06:37.791685Z"}


In [4]:
scored_data_out = model_scoring(scoring_config, graph_data, G, all_nodes_df, final_communities)

{"model_id": "AMM-395", "component": "model_scoring", "pipeline_job": null, "event": "Starting model scoring", "logger": "utils.model_logging.model_logging", "level": "info", "timestamp": "2025-12-03T04:06:37.826465Z"}
Loading data locally: C:\GitHub\CTP_syndicate_model\model_package\model_scoring\trained_models\dominant_AD_model_2025-09-26-23-23.pth
{"model_id": "AMM-395", "component": "model_scoring", "pipeline_job": null, "event": "Model prediction completed", "logger": "utils.model_logging.model_logging", "level": "info", "timestamp": "2025-12-03T04:06:43.146531Z"}
{"model_id": "AMM-395", "component": "model_scoring", "pipeline_job": null, "event": "Output generation completed", "logger": "utils.model_logging.model_logging", "level": "info", "timestamp": "2025-12-03T04:06:45.410885Z"}


In [5]:
edh_upload_data(scoring_config, scored_data_out)

{"model_id": "AMM-395", "component": "data_output", "pipeline_job": null, "event": "Starting data output process", "logger": "utils.model_logging.model_logging", "level": "info", "timestamp": "2025-12-03T04:06:45.458000Z"}
{"model_id": "AMM-395", "component": "data_output", "pipeline_job": null, "event": "Started Sub-task to Extract Scored Data Latest from EDH", "logger": "utils.model_logging.model_logging", "level": "info", "timestamp": "2025-12-03T04:06:45.463886Z"}
{"model_id": "AMM-395", "component": "data_output", "pipeline_job": null, "event": "Started task to Create Output Tables for Upload to EDH", "logger": "utils.model_logging.model_logging", "level": "info", "timestamp": "2025-12-03T04:06:47.446456Z"}
{"model_id": "AMM-395", "component": "data_output", "pipeline_job": null, "event": "Creating Model Outputs Completed", "logger": "utils.model_logging.model_logging", "level": "info", "timestamp": "2025-12-03T04:06:47.476687Z"}
{"model_id": "AMM-395", "component": "data_output",

In [11]:
from utils.model_logging.model_logging import log_utility
from utils.data_postgres.data_postgres import data_postgres

logger = log_utility(model_id = 'AMM-395', component = 'data_output')
def extract_scored_data(db, dbase, schema, obj):

    logger.info('Started Sub-task to Extract Scored Data Latest from EDH')

    exists_latest = data_postgres(db, dbase, fn='exists',  query= None, schema=schema, name = obj)

    if exists_latest:
        scored_data_latest = data_postgres(db, dbase, fn='get',  query = ("select * from %s.%s" %(schema, obj)))
    else: 
        logger.info('Latest data does not exist!')
        scored_data_latest = None

    return scored_data_latest

In [18]:
model_name = scoring_config['model_name']
objs_iadpprod = {'scores':model_name+'_reporting', 
                'scores_history':model_name+'_history', 
                'scores_audit':model_name +'_audit'}
output_env =  scoring_config['output_env']
db_config = DBENVS[output_env]
schema='mod_analytics'
scored_latest  = extract_scored_data(db_config['db'], db_config['dbase'], schema, objs_iadpprod['scores'])

{"model_id": "AMM-395", "component": "data_output", "pipeline_job": null, "event": "Started Sub-task to Extract Scored Data Latest from EDH", "logger": "utils.model_logging.model_logging", "level": "info", "timestamp": "2025-12-03T03:10:28.967330Z"}


In [25]:
scored_data_out['exposure_1_lodgement_date'].dtype

dtype('O')

In [31]:
scored_data_out['exposure_1_lodgement_date'][0]

datetime.date(2023, 8, 10)

In [24]:
scored_latest['exposure_1_lodgement_date'].dtype

dtype('<M8[ns]')

In [32]:
scored_latest['exposure_1_lodgement_date'][0]

Timestamp('2025-05-09 00:00:00')